# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

- Fuite de données (data leakage) : utiliser statistiques/globales calculées sur tout EEG_Features_10s (ou inclure colonnes liées aux labels) permettrait au modèle d’exploiter des informations du jeu de test.
- Variabilité inter‑sujet / échelle : chaque sujet a un niveau de baseline différent ; sans normalisation par baseline le modèle apprendra l’identité/échelle du sujet plutôt que la charge cognitive.
- Baselines et prétraitement nécessaires : les fichiers de baseline servent à calibrer (diviser) les features ; ils ne doivent pas être copiés tels quels comme exemples d’entraînement.
- Standardisation au bon moment : appliquer StandardScaler ou normalisation globale avant la séparation train/test provoque une fuite — il faut ajuster le scaler uniquement sur X_train (mettez-le dans un Pipeline).
- Métadonnées et labels : colonnes comme Participant, File, Window, Level ou Label ne doivent pas être utilisées comme features (elles fugueraient l’appariement avec la cible).
- Robustesse et généralisation : sans normalisation par sujet et validation adaptée (ex. LOSO), les performances sur sujets jamais vus seront surestimées.

In [ ]:
from pathlib import Path

# TODO : adapter ce chemin à votre organisation locale.
BASE_PATH = Path("dataset")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel"]

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

- Réduit la variance inter‑sujet : chaque sujet a un niveau physiologique (échelle) différent ; diviser par la moyenne de baseline évite que le modèle apprenne l’identité du sujet.
- Supprime les décalages de niveau (offsets) liés au matériel ou à l’enregistrement.
- Atténue la non‑stationnarité : compense les dérives lentes du signal entre la baseline et la tâche.
- Améliore la comparabilité des features entre conditions et facilite la détection d’effets liés à la charge cognitive plutôt qu’à l’échelle du signal.
-  Favorise la généralisation et une évaluation réaliste (surtout en LOSO).

In [ ]:
import pandas as pd

def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    feature_cols = [c for c in num_cols if c not in metadata_columns]
    return feature_cols

def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.

    Indications :
    - parcourir les fichiers CSV de feature_dir ;
    - garder uniquement les fichiers dont le nom contient 'baseline' ;
    - lire chaque fichier avec pandas.read_csv ;
    - identifier le Participant avec df['Participant'].iloc[0] ;
    - calculer la moyenne de chaque feature ;
    """
    baseline_avgs = {}
    accum = {}
    feature_dir = Path(feature_dir)

    for csv_path in feature_dir.rglob('*.csv'):
        if 'baseline' not in csv_path.name.lower():
            continue
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue
        if 'Participant' not in df.columns:
            continue

        pid = str(df['Participant'].iloc[0])
        feat_cols = get_feature_columns(df)
        if pid not in accum:
            accum[pid] = []
        if feat_cols:
            accum[pid].append(df[feat_cols])

    for pid, frames in accum.items():
        if not frames:
            continue
        allf = pd.concat(frames, ignore_index=True)
        baseline_avgs[pid] = allf.mean().to_dict()

    return baseline_avgs

def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    if participant_id not in baseline_avgs:
        raise KeyError(f"Aucune baseline disponible pour le participant {participant_id}")

    df_norm = df.copy()
    participant_avg = pd.Series(baseline_avgs[participant_id])
    feature_cols = [c for c in get_feature_columns(df_norm) if c in participant_avg.index]

    if feature_cols:
        df_norm.loc[:, feature_cols] = df_norm.loc[:, feature_cols].div(participant_avg[feature_cols])

    return df_norm

def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    """
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)

    for csv_path in EEG_FEATURE_DIR.glob('*.csv'):

        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        if 'Participant' not in df.columns:
            continue

        participant_id = str(df['Participant'].iloc[0])
        if participant_id not in baseline_avgs:
            continue

        df_norm = normalize_by_baseline(df, participant_id, baseline_avgs)
        output_path = NORMALIZED_EEG_DIR / f"norm_{csv_path.stem}.csv"
        df_norm.to_csv(output_path, index=False)


In [ ]:
run_eeg_normalization()

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse

Les fichiers de baseline ont un rôle de **référence de calibration**, pas de données d'apprentissage. Leur seule utilité est de fournir la moyenne $\mu_{baseline}^{(s,f)}$ qui sert à normaliser les fichiers de tâche. Une fois cette normalisation effectuée, les fenêtres de baseline n'ont pas de label PAAS associé (le sujet est au repos, sans charge cognitive induite) : les inclure dans le dataset supervisé introduirait des exemples non étiquetables qui ne correspondent à aucune condition de charge cognitive. Elles sont donc consommées pendant la normalisation puis exclues du pipeline d'apprentissage.

In [ ]:
# Vérification du dossier Normalized_Features_10s/EEG
import pandas as pd

norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_*.csv"))

if not norm_files:
    print(f"Aucun fichier norm_*.csv trouvé dans {NORMALIZED_EEG_DIR.resolve()}")
    print("Lance d'abord run_eeg_normalization().")
else:
    print(f"{len(norm_files)} fichier(s) dans {NORMALIZED_EEG_DIR}\n")
    print(f"{'Fichier':<45}  {'Lignes':>6}  {'Features':>8}  {'NaN':>5}  {'Participants'}")
    print("-" * 85)

    for f in norm_files:
        df_v = pd.read_csv(f)
        meta = [c for c in df_v.columns if c in METADATA_COLUMNS]
        feat_cols = [c for c in df_v.select_dtypes(include='number').columns if c not in meta]
        n_nan = int(df_v[feat_cols].isna().sum().sum())
        participants = df_v["Participant"].unique().tolist() if "Participant" in df_v.columns else ["?"]
        print(f"  {f.name:<43}  {len(df_v):>6}  {len(feat_cols):>8}  {n_nan:>5}  {participants}")


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [ ]:
import re

def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.
    Ex : filtered_eeg_data_level_3.csv → 3
    Retourne None si non trouvé.
    """
    match = re.search(r"level_(\d+)", file_name)
    return int(match.group(1)) if match else None


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.
    Le niveau est extrait de la colonne 'File' de la ligne (pas du nom du fichier normalisé).
    row       : une ligne du DataFrame, passée automatiquement par df.apply(..., axis=1)
    labels_df : DataFrame du fichier Labels/<Participant>.csv
    """
    # Extraire le level depuis la colonne File de la ligne
    level = extract_level_from_filename(str(row.get("File", "")))
    if level is None:
        return float("nan")

    time_stamp = (int(row["Window"]) + 1) * 10
    label_col  = f"lvl_{level}"

    if label_col not in labels_df.columns:
        return float("nan")

    mask = labels_df["time"] == time_stamp
    if not mask.any():
        return float("nan")

    return labels_df.loc[mask, label_col].values[0]


def attach_labels_eeg():
    """
    Génère les fichiers du dossier Normalized_Features_10s_With_Label/EEG.
    Chaque fichier de sortie contient deux nouvelles colonnes : Level et Label.
    """
    norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_*.csv"))

    if not norm_files:
        print("Aucun fichier norm_*.csv trouvé. Lance d'abord run_eeg_normalization().")
        return

    for csv_path in norm_files:
        df = pd.read_csv(csv_path)

        if "Participant" not in df.columns:
            print(f"[SKIP] colonne Participant manquante : {csv_path.name}")
            continue

        participant_id = str(df["Participant"].iloc[0])
        label_file = LABEL_DIR / f"{participant_id}.csv"

        if not label_file.exists():
            print(f"[SKIP] labels manquants pour {participant_id}")
            continue

        labels_df = pd.read_csv(label_file)

        # Level extrait de la colonne File de chaque ligne (via get_label_for_row)
        df["Level"] = df["File"].apply(extract_level_from_filename)
        df["Label"] = df.apply(get_label_for_row, axis=1, labels_df=labels_df)

        df = df.dropna(subset=["Label"])
        df["Label"] = df["Label"].astype(float)

        out_path = LABELED_EEG_DIR / csv_path.name
        df.to_csv(out_path, index=False)
        print(f"[OK]  {csv_path.name:<50}  {len(df)} lignes labellisées")

In [ ]:
attach_labels_eeg()

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

- Level est le numéro du scénario de conduite (1 à 9), extrait du nom du fichier. C'est un identifiant de condition expérimentale, pas une mesure de charge cognitive.

- Label est le score PAAS associé à cette fenêtre temporelle pour ce niveau, issu du fichier de labels. C'est la cible d'apprentissage, une mesure subjective de la charge cognitive.

On ajoute les deux car Level est nécessaire pour aller chercher la bonne colonne (lvl_<Level>) dans le fichier de labels, et il peut servir de variable de traçabilité. Label est ce qu'on cherche à prédire.

In [ ]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG
import pandas as pd

def verify_labeled_eeg_dir():
    """Affiche un résumé des fichiers norm_*.csv dans LABELED_EEG_DIR.

    Vérifie :
    - présence des colonnes de métadonnées ;
    - nombre de features numériques ;
    - présence des colonnes `Level` et `Label` ;
    - nombre de valeurs manquantes dans les features ;
    - participants présents dans chaque fichier.
    """
    norm_files = sorted(LABELED_EEG_DIR.glob("norm_*.csv"))

    if not norm_files:
        print(f"Aucun fichier norm_*.csv trouvé dans {LABELED_EEG_DIR.resolve()}")
        print("Lance d'abord attach_labels_eeg() pour générer les fichiers labellisés.")
        return

    print(f"{len(norm_files)} fichier(s) dans {LABELED_EEG_DIR}\n")
    print(f"{'Fichier':<45}  {'Lignes':>6}  {'Features':>8}  {'NaN_feat':>8}  {'HasLevel':>8}  {'HasLabel':>8}  {'Participants'}")
    print("-" * 120)

    for f in norm_files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"Erreur lecture {f.name}: {e}")
            continue

        meta = [c for c in df.columns if c in METADATA_COLUMNS]
        # Exclure Level et Label des features numériques
        exclude = set(meta + ['Level', 'Label'])
        feat_cols = [c for c in df.select_dtypes(include=['number']).columns if c not in exclude]
        n_nan_feat = int(df[feat_cols].isna().sum().sum()) if feat_cols else 0
        has_level = 'Level' in df.columns
        has_label = 'Label' in df.columns
        participants = df['Participant'].unique().tolist() if 'Participant' in df.columns else ['?']

        print(f"  {f.name:<43}  {len(df):>6}  {len(feat_cols):>8}  {n_nan_feat:>8}  {str(has_level):>8}  {str(has_label):>8}  {participants}")


# Exécution de la vérification
verify_labeled_eeg_dir()

## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [ ]:
import pandas as pd

def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    norm_files = sorted(LABELED_EEG_DIR.glob("norm_*.csv"))
    if not norm_files:
        print(f"Aucun fichier norm_*.csv trouvé dans {LABELED_EEG_DIR.resolve()}")
        return pd.DataFrame()

    frames = []
    for f in norm_files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"Erreur lecture {f.name}: {e}")
            continue
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)

    out_path = LABELED_EEG_DIR / "combined_labeled_eeg.csv"
    try:
        df_all.to_csv(out_path, index=False)
        print(f"Fichier combiné sauvegardé : {out_path}")
    except Exception as e:
        print(f"Échec sauvegarde combinée : {e}")

    return df_all


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

In [ ]:
# Création des cibles de classification.
if df.empty:
    print("DataFrame vide — exécutez d'abord la génération des fichiers labellisés.")
else:
    # Binaire : 1-4 -> 0 (faible), 5-9 -> 1 (élevée)
    df["Label_Binary"] = df["Label"].apply(lambda v: 0 if v <= 4 else 1)

    # Ternaire : 1-3 faible(0), 4-6 moyenne(1), 7-9 élevée(2)
    def to_ternary(v):
        if v <= 3:
            return 0
        if v <= 6:
            return 1
        return 2

    df["Label_Ternary"] = df["Label"].apply(to_ternary)

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

Les colonnes Participant, File, Window, Level et Label ne doivent pas être incluses dans les features car :

- Participant, File, Window sont des métadonnées d'identification : elles n'ont aucun lien causal avec la charge cognitive, et les inclure ferait apprendre au modèle des identifiants plutôt que des patterns physiologiques.
- Level est corrélé artificiellement avec Label (le label est extrait via le level), ce qui provoquerait une fuite de données : le modèle apprendrait à reconnaître le scénario plutôt que la charge cognitive réelle.
- Label est la cible : l'inclure en feature reviendrait à donner la réponse au modèle.

In [ ]:
# Préparation des données d’entraînement
exclude = set(METADATA_COLUMNS + ["Level", "Label", "Label_Binary", "Label_Ternary"])
feature_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in exclude]

print("Nombre d'exemples :", len(df))
print("Nombre de features numériques :", len(feature_cols))
print("Exemples de features :", feature_cols[:10])

# Vérifications basiques
n_missing = int(df[feature_cols].isna().sum().sum()) if feature_cols else 0
print("Valeurs manquantes (features) :", n_missing)
print("Répartition classes binaires :\n", df["Label_Binary"].value_counts())
print("Répartition classes binaires (fraction) :\n", df["Label_Binary"].value_counts(normalize=True))

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

# Vérification minimale
if 'df' not in globals() or df.empty:
    print("DataFrame vide. Exécute d'abord la génération du dataset labellisé.")
else:
    # Variables d'apprentissage
    X = df[feature_cols]
    y = df["Label_Binary"]

    # Séparation train / test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Modèles classiques
    models = {
        "LDA": LinearDiscriminantAnalysis(),
        "SVM": SVC(kernel="rbf", random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    }

    results = []

    for name, clf in models.items():
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", clf)
        ])

        # Le scaler est ajusté uniquement sur X_train
        pipe.fit(X_train, y_train)

        # Puis appliqué sur X_test automatiquement
        y_pred = pipe.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append({
            "Model": name,
            "Accuracy": acc,
            "F1-score": f1
        })

        print(f"\n=== {name} ===")
        print(f"Accuracy : {acc:.4f}")
        print(f"F1-score : {f1:.4f}")
        print(classification_report(y_test, y_pred))

    results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
    print("\nRésumé des résultats :")
    display(results_df)

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

En 10-fold, les données sont mélangées aléatoirement avant d'être réparties en folds. Des fenêtres du même sujet se retrouvent donc à la fois en entraînement et en test : le modèle a déjà "vu" la signature EEG de ce sujet, ce qui facilite la prédiction.

En LOSO, le sujet de test est entièrement absent de l'entraînement. Le modèle doit généraliser à un individu dont il ne connaît pas les caractéristiques physiologiques. Or les signaux EEG varient considérablement d'un sujet à l'autre (morphologie des potentiels, niveaux d'activité de base), ce qui rend la généralisation inter-sujets intrinsèquement plus difficile.

Le LOSO est donc une évaluation plus réaliste d'un système déployé sur un nouveau conducteur, mais produit mécaniquement des performances plus faibles.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, LeaveOneGroupOut
import numpy as np
import pandas as pd

# groups = identifiant du sujet pour chaque ligne, utilisé par LOSO
participants = df["Participant"].values

# Reconstruction des pipelines (même scaler + même clf qu'en section 8)
PIPELINES = {
    name: Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    for name, clf in models.items()
}

# ── 10-fold stratifié ──────────────────────────────────────────────────────
# StratifiedKFold garantit que chaque fold a la même proportion de classes 0/1.
# shuffle=True mélange les lignes avant découpage, évitant les biais de tri.
# cross_val_score entraîne et évalue le pipeline complet sur chaque fold ;
# le StandardScaler est ajusté uniquement sur les 9 folds d'entraînement.

print("=" * 60)
print(" 10-fold cross-validation (stratifié)")
print("=" * 60)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv10_results = {}

for name, pipe in PIPELINES.items():
    scores = cross_val_score(pipe, X, y, cv=kf, scoring="f1_weighted", n_jobs=-1)
    cv10_results[name] = scores
    print(f"  {name:<14}  F1 = {scores.mean():.4f}  ±  {scores.std():.4f}")

# ── Leave-One-Subject-Out (LOSO) ───────────────────────────────────────────
# LeaveOneGroupOut utilise la colonne Participant comme groupe.
# À chaque itération, tous les exemples d'un sujet sont dans le test set,
# tous les autres sujets constituent le train set.
# Résultat : 21 folds (un par sujet), ce qui peut prendre quelques minutes.

print()
print("=" * 60)
print(" Leave-One-Subject-Out (LOSO)")
print("=" * 60)

logo = LeaveOneGroupOut()
loso_results = {}

for name, pipe in PIPELINES.items():
    scores = cross_val_score(pipe, X, y, cv=logo, groups=participants,
                             scoring="f1_weighted", n_jobs=-1)
    loso_results[name] = scores
    print(f"  {name:<14}  F1 = {scores.mean():.4f}  ±  {scores.std():.4f}")

# ── Tableau comparatif ─────────────────────────────────────────────────────
# Δ = F1 (10-fold) - F1 (LOSO) : un Δ élevé indique que le modèle mémorise
# la signature EEG des sujets vus à l'entraînement plutôt que de généraliser.

print()
summary = []
for name in PIPELINES:
    f1_cv   = cv10_results[name].mean()
    f1_loso = loso_results[name].mean()
    summary.append({"Modèle": name, "10-fold F1": f1_cv,
                    "LOSO F1": f1_loso, "Δ (chute)": f1_cv - f1_loso})

summary_df = pd.DataFrame(summary).sort_values("LOSO F1", ascending=False)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

Une seule prédiction peut être erronée à cause d'un artefact EEG ou d'une fluctuation passagère du signal. De plus, la charge cognitive varie naturellement sur de courtes durées, donc une fenêtre de 10 s ne reflète pas forcément un état soutenu. Déclencher une alerte trop tôt risque aussi de perturber le conducteur et d'aggraver la situation. Il vaut mieux attendre plusieurs prédictions consécutives en classe "charge élevée" (par exemple avec une fenêtre glissante) avant de réagir.

In [ ]:
def decision_system(...):
    """
    Transforme les prédictions en décision d'adaptation.
    
    """




## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse

La charge cognitive n'a pas une signature unique et universelle : elle se manifeste différemment selon les individus et les situations. L'EEG seul capte l'activité corticale, mais ne dit rien de la réponse du système nerveux autonome (ECG), du stress physiologique (EDA) ou des stratégies de regard du conducteur (Gaze). En combinant ces modalités, on couvre mieux l'ensemble du phénomène.

Cela rend aussi le système plus robuste : si une électrode EEG est momentanément bruitée par un mouvement de tête, les autres capteurs continuent à informer la décision.

La multimodalité aide aussi à mieux généraliser entre sujets. Certains individus ont une réponse EEG marquée mais une EDA quasi-stable ; pour d'autres c'est l'inverse. Un modèle qui n'a accès qu'à une seule modalité rate forcément ces profils atypiques, ce qui pénalise surtout l'évaluation LOSO.